In [ ]:
!pip install cplex 
!pip install docplex
import cplex

In [9]:
from docplex.mp.model import Model

# -----------------------
# Data
# -----------------------
# costs for x and y (3x3)
cx = [[5, 4, 3],
      [3,10, 1],
      [3, 6, 2]]

cy = [[5, 5, 5],
      [5, 5, 5],
      [5, 5, 5]]

# supplies (rows) and demands (cols)
supply  = [3, 2, 2]   # rows i = 1..3
demand  = [1, 2, 4]   # cols j = 1..3

cap = 4               # x_ij <= cap * y_ij

# -----------------------
# Model
# -----------------------
mdl = Model('fixed_charge_transport_3x3')

I, J = range(3), range(3)

# decision variables
x = {(i,j): mdl.continuous_var(lb=0, name=f"x_{i+1}{j+1}") for i in I for j in J}
y = {(i,j): mdl.binary_var(           name=f"y_{i+1}{j+1}") for i in I for j in J}

# objective: min sum c_x * x + sum c_y * y
mdl.minimize(mdl.sum(cx[i][j]*x[i,j] for i in I for j in J) +
             mdl.sum(cy[i][j]*y[i,j] for i in I for j in J))

# row (supply) constraints: sum_j x_ij = supply_i
for i in I:
    mdl.add_constraint(mdl.sum(x[i,j] for j in J) == supply[i], ctname=f"supply_{i+1}")

# column (demand) constraints: sum_i x_ij = demand_j
for j in J:
    mdl.add_constraint(mdl.sum(x[i,j] for i in I) == demand[j], ctname=f"demand_{j+1}")

# linking (capacity) constraints: x_ij <= cap * y_ij
for i in I:
    for j in J:
        mdl.add_constraint(x[i,j] <= cap * y[i,j], ctname=f"link_{i+1}{j+1}")

# -----------------------
# Solve
# -----------------------
sol = mdl.solve(log_output=True)  # set to False to silence

if sol:
    print("\nObjective value:", sol.objective_value)
    print("\nFlows x_ij:")
    for i in I:
        for j in J:
            val = x[i,j].solution_value
            if val > 1e-6:
                print(f"  x_{i+1}{j+1} = {val:.6g}")
    print("\nOpen arcs y_ij:")
    for i in I:
        for j in J:
            if y[i,j].solution_value > 0.5:
                print(f"  y_{i+1}{j+1} = 1")
else:
    print("No solution found. Status:", mdl.get_solve_status())


ModuleNotFoundError: No module named 'docplex'